# 07 — Historical V3 Model Evaluation

> **Notebook status: historical V3 evaluation summary.** The original V3 per-image prediction export and ROC coordinates were not recovered. The metrics and reconstructed confusion matrix are preserved as historical results. The current application uses V4; see Notebook 09 for V4 results.


In [ ]:
import pandas as pd

v3_metrics = pd.DataFrame([
    {"metric":"Accuracy", "validation":0.9784, "test":0.9408602150537635},
    {"metric":"Balanced accuracy", "validation":0.9787, "test":0.9410268270120259},
    {"metric":"AI precision", "validation":1.0000, "test":0.9560439560439561},
    {"metric":"AI recall", "validation":0.9574, "test":0.925531914893617},
    {"metric":"AI F1", "validation":0.9783, "test":0.9405405405405406},
    {"metric":"ROC-AUC", "validation":0.9964, "test":0.995259019426457},
])
display(v3_metrics.style.format({"validation":"{:.2%}", "test":"{:.2%}"}))
print("Threshold:", 0.47)
print("Loaded images: train=872, validation=185, test=186")


## Test confusion matrix

The reported test precision, recall, specificity implied by balanced accuracy, and test support are mutually consistent with the following counts (rows = actual, columns = predicted): **88 real correctly classified, 4 real classified AI-generated, 7 AI-generated classified real, 87 AI-generated correctly classified**. This produces the recorded test accuracy 175/186 = 94.09%, AI precision 87/91 = 95.60%, AI recall 87/94 = 92.55%, and balanced accuracy 94.10%.

This figure/table is a reconstruction from the recorded summary fractions, not a recovered original confusion-matrix export. The original per-image V3 prediction file was not available.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

# Reconstructed from the recorded V3 test metrics; rows actual, columns predicted.
cm_test_reconstructed = np.array([[88, 4], [7, 87]])
display = ConfusionMatrixDisplay(cm_test_reconstructed, display_labels=["Real", "AI-generated"])
display.plot(values_format="d", colorbar=False)
plt.title("V3 test confusion matrix — reconstructed from recorded metrics")
plt.tight_layout(); plt.show()


## ROC curve and per-image error analysis

ROC-AUC is recorded as 99.64% on validation and 99.53% on test (using the exact serialized metric value 0.995259... for the test result). AUC is one summary number; it does not contain enough information to reconstruct the full ROC curve. The original per-image V3 scores are not packaged, so no ROC curve is drawn from invented coordinates.

When the true labels and saved V3 sigmoid scores are available, use the code below to generate the curve and review false positives/false negatives.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score, classification_report, confusion_matrix

# Expected CSV columns: true_label_id (0/1), cnn_ai_score_uncalibrated (0..1), image_path.
scores_path = Path("reports/v3_test_predictions.csv")
if scores_path.exists():
    scores = pd.read_csv(scores_path)
    y_true = scores["true_label_id"].astype(int).to_numpy()
    y_score = scores["cnn_ai_score_uncalibrated"].astype(float).to_numpy()
    y_pred = (y_score >= 0.47).astype(int)
    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc = roc_auc_score(y_true, y_score)
    plt.plot(fpr, tpr, label=f"CNN ROC (AUC={auc:.4f})")
    plt.plot([0,1],[0,1], linestyle="--", label="Chance")
    plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
    plt.title("V3 CNN test ROC curve"); plt.legend(); plt.grid(alpha=0.2); plt.show()
    print(classification_report(y_true, y_pred, target_names=["Real","AI-generated"], zero_division=0))
    scores["predicted_label_id"] = y_pred
    print("False positives:")
    display(scores[(scores.true_label_id == 0) & (scores.predicted_label_id == 1)].head(20))
    print("False negatives:")
    display(scores[(scores.true_label_id == 1) & (scores.predicted_label_id == 0)].head(20))
else:
    print("No per-image V3 prediction CSV found. ROC curve and image-level error analysis cannot be reconstructed from AUC alone.")


## Limitations that must accompany the results

1. V3 used 872 training, 185 validation and 186 test images because Keras' directory loader skipped 2/3/2 WebP files respectively.
2. The 94.09% test accuracy is based on 186 loaded images, not the full expected 188-image split.
3. Source groups overlap across the splits; the evaluation is not a held-out-generator test.
4. The test score does not demonstrate generalisation to completely unseen generators or future generation systems.
5. Authenticity labels are inherited from source labels/manifests and were not independently verified for every file.
6. The score is uncalibrated and is not a fraud probability.
7. The model's output cannot verify a vehicle, seller, listing or ownership claim.

For a stronger next evaluation, use a larger independently reviewed dataset, decode all image formats consistently, group splits by source/vehicle/generator, and hold out entire generator families before any model selection.